In [ ]:
import pandas as pd

df = pd.read_csv("/RetailCleaned.csv", header=None)
df.columns = ['Invoice', 'StockCode', 'Description','Quantity', 'InvoiceDate', 'Price', 'Customer_Id', 'Country', 'SourceSheet']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info
df.head()


/tmp/ipykernel_1141/2397196548.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/RetailCleaned.csv", header=None)


(1033036, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_Id,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011


In [ ]:
# Filtering and calculating Recency, Freq, Monetary
rfm_data = df[(df['Customer_Id'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

# Create a TotalSum column to simplify the Monetary calculation
rfm_data['TotalSum'] = rfm_data['Quantity'] * rfm_data['Price']

rfm = rfm_data.groupby('Customer_Id').agg(
    Recency = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary = ('TotalSum', 'sum')
).reset_index()

display(rfm.head())


,Customer_Id,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [ ]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm['IsChurned'] = rfm['Recency'] > 90

In [ ]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[
    (df['Customer_Id'].notna()) & (df['Quantity'] > 0)
].copy()

df['InvoiceMonth'] = (
    df['InvoiceDate']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[
        [
            'Customer_Id',
            'Recency',
            'Frequency',
            'Monetary',
            'R_Score',
            'F_Score',
            'M_Score',
            'Segment',
            'IsChurned'
        ]
    ],
    on='Customer_Id',
    how='left'
)

df_final.to_excel(
    'EcommerceDashboardData.xlsx',
    index=False
)